### 1 - Load Data

In [6]:
%pip install annoy

Note: you may need to restart the kernel to use updated packages.


In [7]:
import numpy as np
import pandas as pd

# load vector
X = np.load("../0_data/processed/features.npy")

# load metadata (berisi lagu dan penyanyi)
meta = pd.read_csv("../0_data/processed/meta.csv")

# memeriksa sinkronisasi data dan metadata
assert len(X) == len(meta) 

print(X.shape)
meta.head()

(232725, 9)


,track_name,artist_name,genre
0,C'est beau de faire un Show,Henri Salvador,Movie
1,Perdu d'avance (par Gad Elmaleh),Martin & les fées,Movie
2,Don't Let Me Be Lonely Tonight,Joseph Williams,Movie
3,Dis-moi Monsieur Gordon Cooper,Henri Salvador,Movie
4,Ouverture,Fabien Nataf,Movie


### 2 - Build Annoy Index

In [8]:
def build_annoy(X, n_trees=30):
    from annoy import AnnoyIndex
    import time
    
    dim = X.shape[1]
    index = AnnoyIndex(dim, 'angular')
    
    start = time.time()
    for i, vec in enumerate(X):
        index.add_item(i, vec)
    index.build(n_trees)
    build_time = time.time() - start
    
    return index, build_time

### 3 - Query Function

In [9]:
def query_annoy(index, X, query_idx, k=5):
    import time
    
    query_vector = X[query_idx]
    
    start = time.time()
    indices, distances = index.get_nns_by_vector(
        query_vector, k, include_distances=True
    )
    query_time = time.time() - start

    if query_idx >= len(X):
        raise ValueError("query_idx melebihi jumlah data")
    
    return indices, distances, query_time

### 4 - Hasil Rekomendasi

In [12]:
def run_annoy(index, X, meta, query_idx, build_time, k=20):
    indices, distances, query_time = query_annoy(index, X, query_idx, k)
    
    print("=== Annoy (Filtered) ===")
    print(f"Build time: {build_time:.4f} sec")
    print(f"Query time: {query_time:.6f} sec")
    
    print("\n🎵 Query:")
    print(
        meta.iloc[query_idx]["track_name"], "-",
        meta.iloc[query_idx]["artist_name"], "-",
        meta.iloc[query_idx]["genre"]
    )
    
    query_genre = meta.iloc[query_idx]["genre"]
    
    print("\n🎧 Recommendations (Same Genre):")
    
    count = 0
    for i in indices:
        if i == query_idx:
            continue
        
        if meta.iloc[i]["genre"] != query_genre:
            continue
        
        print(
            f"- {meta.iloc[i]['track_name']} - "
            f"{meta.iloc[i]['artist_name']} - "
            f"{meta.iloc[i]['genre']} "
            f"({distances[indices.index(i)]:.4f})"
        )
        
        count += 1
        if count >= k - 1:
            break
    
    # fallback kalau kosong
    if count == 0:
        print("\n(No same-genre results, showing closest matches)\n")
        
        for i in indices:
            if i == query_idx:
                continue
            print(
                f"- {meta.iloc[i]['track_name']} - "
                f"{meta.iloc[i]['artist_name']} - "
                f"{meta.iloc[i]['genre']} "
                f"({distances[indices.index(i)]:.4f})"
            )
    
    return {
        "build_time": build_time,
        "query_time": query_time,
        "indices": indices,
        "distances": distances
    }

# Build sekali
index, build_time = build_annoy(X)

# Run
result = run_annoy(index, X, meta, query_idx=10, build_time=build_time)

=== Annoy (Filtered) ===
Build time: 4.0975 sec
Query time: 0.000000 sec

🎵 Query:
Symphony No.4 In E Minor Op.98 : IV. Allegro Energico E Passionato - Leopold Stokowski - Movie

🎧 Recommendations (Same Genre):
- On A Dark Night - Alan Menken - Movie (0.0864)
- The Gifts of Beauty and Song / Maleficent Appears / True Love Conquers All - From "Sleeping Beauty"/Soundtrack Version - Chorus - Sleeping Beauty - Movie (0.0969)
